# Lab 7 · Notebook 1: Finding the suspects

**Time:** about 35 minutes. **What you will do:** go from a table of 1,165 cancer cell lines to one gene that skin cancer needs and other cancers do not.

There are no plots in this notebook. A figure should answer a question you already have, so this notebook finds the question and the answer first. Notebook 2 draws it.

The notebook works like the ones in Lab 6: run the **Demo** cells, write code in the **Write it** cells and run the self-check underneath, and fix the **Broken** cells.

## The data

This is the DepMap table from the lecture. Researchers grew 1,165 cancer cell lines and used CRISPR to switch off one gene at a time. Each number is a **gene effect**: how badly the cells grew without that gene.

- around `0`: the cells did not need the gene
- around `-1`: losing the gene hurt the cells badly
- `-2` or lower: the cells could hardly survive without it

**One row is one cell line.** `oncotree_lineage` is the tissue the cancer came from (Skin, Lung, Bowel and so on) and `oncotree_primary_disease` is the type of cancer. The other 18 columns, from `SOX10` to `AADAC`, are gene effects.

In [ ]:
# Run this first. It loads the DepMap table from the lecture.
import urllib.request
from pathlib import Path

import pandas as pd

FILE = "depmap.csv"
URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab07/data/" + FILE

# On Colab the file isn't there yet, so download it once.
if not Path(FILE).exists():
    urllib.request.urlretrieve(URL, FILE)

df = pd.read_csv(FILE)
print("Loaded", FILE, df.shape)

## 1. Look before you analyse

The same first step as Lab 6: before you ask the table anything, find out how big it is and how many rows fall into each group.

`value_counts()` counts how many rows carry each label. It gives back a Series whose index is the labels, so you can look up one label with square brackets.

In [ ]:
# DEMO - how many cell lines does each lineage have?
counts = df["oncotree_lineage"].value_counts()
print(counts.head(8))
print()
print("lineages:", len(counts))
print("lung lines:", counts["Lung"])

### Write it

**Your task:**

1. Store the number of **skin** cell lines in `n_skin`.
2. Store the number of lineages that have **fewer than 10** cell lines in `n_small`. (Hint: `counts < 10` is a mask, and `.sum()` counts the `True` values.)

In [ ]:
# TODO - skin lines, and small lineages
n_skin = counts["Skin"]
n_small = (counts < 10).sum()

# --- self-check (do not edit) ---
for name in ["n_skin", "n_small"]:
    assert name in globals(), f"Make a variable called {name}"
assert n_skin == 73, f"n_skin should be 73, got {n_skin}. Look up 'Skin' in counts"
assert n_small == 6, f"n_small should be 6, got {n_small}. Use (counts < 10).sum()"
print("Correct: 73 skin lines, and 6 lineages with fewer than 10 lines")

Seventy-three skin lines is enough to say something about skin. A lineage with three lines is not: one odd cell line would change its average a lot. Check the size of a group before you trust a result about it.

## 2. Skin, and everything else

You filtered rows with a mask in Lab 6. Here you need two tables: the skin lines, and all the lines that are **not** skin. The second one uses `!=`, which means "not equal".

Keep the second table. Most questions in this lab compare skin with the rest, and you cannot compare with half the data.

In [ ]:
# DEMO - the lung lines, and everything that is not lung
lung = df[df["oncotree_lineage"] == "Lung"]
not_lung = df[df["oncotree_lineage"] != "Lung"]
print("lung:", len(lung), "  not lung:", len(not_lung), "  together:", len(lung) + len(not_lung))

### Write it

**Your task:** make a table `skin` with the skin cell lines, and a table `rest` with every other cell line.

In [ ]:
# TODO - skin and the rest
skin = df[df["oncotree_lineage"] == "Skin"]
rest = df[df["oncotree_lineage"] != "Skin"]

# --- self-check (do not edit) ---
for name in ["skin", "rest"]:
    assert name in globals(), f"Make a variable called {name}"
assert len(skin) == 73, f"skin should have 73 rows, got {len(skin)}. Check the spelling: 'Skin' with a capital S"
assert len(rest) == 1092, f"rest should have 1,092 rows, got {len(rest)}. Use != 'Skin'"
assert len(skin) + len(rest) == len(df), "skin and rest together should make up the whole table"
print("Correct: 73 + 1,092 = 1,165 cell lines")

### Write it

A gene effect below `-1` means the cells were hurt badly. Two conditions at once need `&` and a pair of round brackets around each condition, as in Lab 6.

**Your task:**

1. Store the number of skin lines with `SOX10` below `-1` in `n_need_sox10`.
2. Store the number of skin lines with `SOX10` below `-1` **and** `BRAF` below `-0.5` in `n_need_both`.

In [ ]:
# TODO - how many skin lines need SOX10? And SOX10 and BRAF?
n_need_sox10 = len(skin[skin["SOX10"] < -1])
n_need_both = len(skin[(skin["SOX10"] < -1) & (skin["BRAF"] < -0.5)])

# --- self-check (do not edit) ---
for name in ["n_need_sox10", "n_need_both"]:
    assert name in globals(), f"Make a variable called {name}"
assert n_need_sox10 == 52, f"n_need_sox10 should be 52, got {n_need_sox10}. Filter skin (not df) with skin['SOX10'] < -1"
assert n_need_both == 42, f"n_need_both should be 42, got {n_need_both}. Two conditions, each in round brackets, joined with &"
print("Correct: 52 skin lines need SOX10, and 42 of them need BRAF as well")

## 3. Which genes do the skin lines need most?

To rank the genes, take the mean of every gene column and look at the most negative ones. Most negative means most needed.

```python
means = skin.mean(numeric_only=True)
means.nsmallest(5)
```

`numeric_only=True` tells pandas to skip the text columns, such as the cell line names. `nsmallest(5)` gives the five smallest values, smallest first.

### Broken

This should give the mean of every gene in the lung lines. Run it and read the last line of the error.

In [ ]:
# BROKEN - the mean of every gene in lung
lung_mean = lung.mean(numeric_only=True)
lung_mean.head()


# --- self-check (do not edit) ---
assert len(lung_mean) == 18, f"lung_mean should have one value per gene (18), got {len(lung_mean)}"
print("Fixed. pandas cannot take the mean of a cell line name. numeric_only=True skips the text columns.")

### Write it

**Predict first:** which kind of gene do you expect at the top of the list for skin cancer?

**Your task:**

1. Store the mean of every gene in the skin lines in `skin_mean`.
2. Store the five most negative values in `skin_top5`.

In [ ]:
# TODO - the five genes skin lines need most
skin_mean = skin.mean(numeric_only=True)
skin_top5 = skin_mean.nsmallest(5)
skin_top5

# --- self-check (do not edit) ---
for name in ["skin_mean", "skin_top5"]:
    assert name in globals(), f"Make a variable called {name}"
assert len(skin_mean) == 18, f"skin_mean should have 18 values, one per gene, got {len(skin_mean)}"
assert list(skin_top5.index) == ["RAN", "SNRPD1", "RPL5", "PSMA1", "RPS6"], f"skin_top5 has {list(skin_top5.index)}. Use skin_mean.nsmallest(5)"
print("Correct: RAN first, at", round(skin_top5["RAN"], 2))

RAN, at about -3.9. That looks like a result. Before you believe it, run the obvious control: ask the same question of cancers that have nothing to do with skin.

In [ ]:
# DEMO - the same question for four lineages
for name in ["Skin", "Lung", "Bowel", "Breast"]:
    lineage = df[df["oncotree_lineage"] == name]
    top = lineage.mean(numeric_only=True).nsmallest(4)
    print(name, list(top.index))

The four lists are almost the same: RAN first every time, then SNRPD1 and RPL5.

These are **housekeeping genes**. RAN moves proteins in and out of the nucleus, RPL5 and RPS6 are part of the ribosome, PSMA1 is part of the proteasome and SNRPD1 helps splice RNA. Every living cell needs them, cancer or not. A drug against RAN would kill the melanoma, and the patient's healthy cells with it.

So "which gene does skin need most?" was the wrong question. A better one is:

> **Which gene does skin need more than the other cancers do?**

## 4. A difference finds what is specific

To answer the better question, take the mean in skin and subtract the mean in the rest. A Series minus a Series lines the values up by their labels, so each gene is subtracted from itself.

- a large **negative** difference: the gene is needed in skin much more than elsewhere
- a difference near **zero**: the gene is needed the same everywhere, like RAN

`idxmin()` gives the **label** of the smallest value, rather than the value itself.

In [ ]:
# DEMO - which gene does lung need more than the rest?
lung_diff = lung.mean(numeric_only=True) - not_lung.mean(numeric_only=True)
print(lung_diff.nsmallest(3).round(2))
print()
print("most lung-specific:", lung_diff.idxmin())

KRAS: many lung cancers carry a KRAS mutation, and those lines need KRAS to keep growing.

### Write it

**Your task:**

1. Store the mean of every gene in `rest` in `rest_mean`.
2. Store `skin_mean - rest_mean` in `difference`.
3. Store the name of the most skin-specific gene in `top_gene`.

In [ ]:
# TODO - needed in skin more than elsewhere
rest_mean = rest.mean(numeric_only=True)
difference = skin_mean - rest_mean
top_gene = difference.idxmin()
print(difference.nsmallest(5).round(2))

# --- self-check (do not edit) ---
for name in ["rest_mean", "difference", "top_gene"]:
    assert name in globals(), f"Make a variable called {name}"
assert round(difference["SOX10"], 2) == -1.06, f"difference['SOX10'] should be -1.06, got {round(difference['SOX10'], 2)}. Subtract in the order skin_mean - rest_mean"
assert top_gene == "SOX10", f"top_gene should be 'SOX10', got {top_gene!r}. Use difference.idxmin()"
print("Correct: SOX10. RAN's difference is only", round(difference["RAN"], 2))

SOX10 comes first, then BRAF. RAN, which topped every raw list, is now close to zero, because the skin lines and the other lines need it equally.

This idea works far beyond this table. A raw ranking finds the genes that are **strongest**. A difference finds the genes that are **specific**, and a specific gene is the kind you could aim a drug at.

## 5. Write it once, use it for any lineage

You have now done the same steps for lung and for skin. Lab 4 says what to do with steps you repeat: put them in a function.

A function should work from its **arguments only**. Inside `most_specific`, use `table` and `lineage`, never `df` or `skin`. Then it gives the right answer for any table and any lineage you pass in.

### Write it

**Your task:** complete `most_specific(table, lineage)`. It should return the name of the gene that `lineage` needs most compared with every other lineage in `table`.

1. Make `inside` (rows of that lineage) and `outside` (all other rows) from `table`.
2. Subtract the two means.
3. Return the label of the smallest difference.

In [ ]:
# TODO - complete the function
def most_specific(table, lineage):
    """Return the gene that `lineage` needs most, compared with all other lineages."""
    inside = table[table["oncotree_lineage"] == lineage]
    outside = table[table["oncotree_lineage"] != lineage]
    diff = inside.mean(numeric_only=True) - outside.mean(numeric_only=True)
    return diff.idxmin()

# --- self-check (do not edit) ---
assert most_specific(df, "Skin") == "SOX10", f"most_specific(df, 'Skin') should be 'SOX10', got {most_specific(df, 'Skin')!r}"
assert most_specific(df, "Bowel") == "CTNNB1", f"most_specific(df, 'Bowel') should be 'CTNNB1', got {most_specific(df, 'Bowel')!r}. Does your function use table and lineage, and not skin?"
small = df[df["oncotree_lineage"].isin(["Pancreas", "Skin", "Bowel"])]
assert most_specific(small, "Pancreas") == "KRAS", "most_specific gave the wrong gene on a smaller table. Use table inside the function, not df"
print("Correct: SOX10 for skin, CTNNB1 for bowel, KRAS for pancreas")

### Write it

**Your task:** use your function in a loop. For each lineage in `to_test`, store its most specific gene in the dictionary `specific`, with the lineage as the key.

Then look the genes up. PAX8 and MYB are **lineage genes**: they switch on the programme that makes a cell a kidney cell or a blood cell, and the cancer still depends on that programme.

In [ ]:
# TODO - one gene per lineage
to_test = ["Skin", "Bowel", "Ovary/Fallopian Tube", "Kidney", "Myeloid", "Pancreas"]
specific = {}
for lineage in to_test:
    specific[lineage] = most_specific(df, lineage)
specific

# --- self-check (do not edit) ---
expected = {"Skin": "SOX10", "Bowel": "CTNNB1", "Ovary/Fallopian Tube": "PAX8",
            "Kidney": "PAX8", "Myeloid": "MYB", "Pancreas": "KRAS"}
assert specific == expected, f"specific is {specific}. Loop over to_test and store most_specific(df, lineage) under specific[lineage]"
print("Correct:", specific)

## 6. Does the mean tell you enough?

One more check before you draw anything, because it changes what the figure needs to show.

The mean SOX10 effect in skin is about -1.3. But a mean is one number standing in for 73 cell lines. Do they all need SOX10, or do some need it a lot and some not at all?

### Write it

**Your task:** store the number of skin lines with `SOX10` **above** `-0.5` (they hardly need SOX10) in `n_barely`.

In [ ]:
# TODO - skin lines that hardly need SOX10
n_barely = len(skin[skin["SOX10"] > -0.5])

# --- self-check (do not edit) ---
assert "n_barely" in globals(), "Make a variable called n_barely"
assert n_barely == 18, f"n_barely should be 18, got {n_barely}. Filter skin with skin['SOX10'] > -0.5"
print("Correct: 18 of the 73 skin lines hardly need SOX10")

So "skin cancer needs SOX10" is true for most skin lines and false for 18 of them, about a quarter.

That decides the plot. A bar showing the mean would hide those 18 lines completely. The figure in notebook 2 shows **every cell line as a dot**, so the reader can see the split. Notebook 3 finds out who the 18 are.

## If you finish early

1. Run `difference.nlargest(3)`. A large **positive** difference means skin needs the gene *less* than other cancers do. Which genes are they? KRAS is one: most melanomas are driven by BRAF or NRAS, not KRAS.
2. Use `most_specific` on every lineage with at least 30 cell lines (`counts[counts >= 30].index`). Which lineages come out with a housekeeping gene? What does that tell you about this table of 18 genes?
3. Change `most_specific` so it returns the gene **and** its difference, rounded to 2 decimal places, as a tuple.

In [ ]:
# Your code here

## Recap

You can now:

- check the size of each group with `value_counts()` before trusting a result about it
- make a table and its complement with `==` and `!=`
- rank columns with `mean(numeric_only=True)` and `nsmallest()`
- spot the housekeeping trap: the strongest dependency is often the least interesting one
- use a **difference** to find what is specific, and `idxmin()` to name it
- put repeated steps in a function that works from its arguments only

**Next:** `02_showing_it`, where SOX10 becomes a figure.